### 채널 내 영상 url 추출

In [9]:
import yt_dlp

channel_url = 'https://www.youtube.com/@bimirya/videos'

# 1. 목록의 URL 리스트만 1초 만에 플랫하게 가져오기
flat_opts = {
    'extract_flat': 'in_playlist',
    'quiet': True,
    'no_warnings': True
}

with yt_dlp.YoutubeDL(flat_opts) as ydl:
    channel_info = ydl.extract_info(channel_url, download=False)
    entries = [e for e in channel_info.get('entries', []) if e]

print(f"전체 영상 {len(entries)}개 수집 완료. 이진 탐색으로 2022년 경계점을 찾습니다...")

detail_opts = {
    'skip_download': True,
    'extract_flat': False,
    'quiet': True,
    'no_warnings': True
}

def get_upload_date(ydl, video_id):
    """영상 ID로 upload_date (YYYYMMDD) 가져오기"""
    try:
        info = ydl.extract_info(f"https://www.youtube.com/watch?v={video_id}", download=False)
        return info.get('upload_date', '')
    except:
        return ''

# 2. 이진 탐색으로 2022년 1월 1일 이전 영상이 처음 나오는 인덱스 찾기
left = 0
right = len(entries) - 1
cutoff_index = len(entries)  # 2022년 이전 영상이 시작되는 첫 인덱스

with yt_dlp.YoutubeDL(detail_opts) as ydl:
    while left <= right:
        mid = (left + right) // 2
        date = get_upload_date(ydl, entries[mid]['id'])
        title = entries[mid].get('title', '')
        print(f"탐색 중 -> [{mid}번째 영상] 날짜: {date} | {title}")

        if date and date >= '20260101':
            # 2022년 이후 영상이면 더 과거(오른쪽)를 확인
            left = mid + 1
        else:
            # 2022년 이전 영상이면 경계 후보로 두고 더 최신(왼쪽)을 확인
            cutoff_index = mid
            right = mid - 1

# 3. 0부터 cutoff_index - 1까지 슬라이싱하여 저장
valid_entries = entries[:cutoff_index]

with open('../data/urls/channel_video_urls.txt', 'w', encoding='utf-8') as f:
    for entry in valid_entries:
        f.write(f"https://www.youtube.com/watch?v={entry['id']}\n")

print(f"\n완료! 전체 {len(entries)}개 중 2022년~현재 영상 총 {len(valid_entries)}개가 'channel_video_urls.txt'에 저장되었습니다.")

전체 영상 536개 수집 완료. 이진 탐색으로 2022년 경계점을 찾습니다...
탐색 중 -> [267번째 영상] 날짜: 20231029 | 작은 마을 하나를 섭렵해 보았습니다 [마쓰야마 투어 2편][EN]ㅣ오마카세, 프렌치 레스토랑
탐색 중 -> [133번째 영상] 날짜: 20250223 | A Chinese restaurant in Busan that also experienced the Korean War😲 The taste of the food that Se...
탐색 중 -> [66번째 영상] 날짜: 20260208 | Michelin “Reservation Hell” Dim Sum— But Is It Worth It? | Summer Palace
탐색 중 -> [99번째 영상] 날짜: 20251016 | Secret Boutique x Blue Ribbon Week 2025 | The Starting Point of Seoul's Culinary Scene
탐색 중 -> [82번째 영상] 날짜: 20251214 | 🍶 Chungmuro ​​Korean Restaurant Perfect for Daytime Drinking | Jingogae | Gopchang Jeon-gol
탐색 중 -> [74번째 영상] 날짜: 20260111 | Only 20 Teams a Day? Colosseum-View Luxury Hotel Review⭐ | Palazzo Manfredi
탐색 중 -> [78번째 영상] 날짜: 20251228 | Don't go all the way to Japan for sushi. | Sushi Kanesaka | Hinotsuki | Michelin 2-star
탐색 중 -> [76번째 영상] 날짜: 20260104 | Took on a 2kg Florence Steak 🥩 (feat. Tripe Burger) | Paoli 1827, Da Nerbone, Caffè Gilli
탐색 중 -> [77번째 영상] 날짜: 20251231

In [10]:
import os
import yt_dlp

input_file = "../data/urls/channel_video_urls.txt"
valid_output_file = "../data/urls/filtered_channel_video_urls.txt"      # 통과된 URL 저장 파일
excluded_output_file = "../data/urls/excluded_channel_video_urls.txt"  # 제외된 URL 저장 파일

# 1. 파일에서 URL 목록 읽어오기
if not os.path.exists(input_file):
    print(f"오류: {input_file} 파일이 없습니다.")
else:
    with open(input_file, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    print(f"총 {len(urls)}개의 URL을 확인합니다...\n")

    ydl_opts = {
        'skip_download': True,
        'extract_flat': False,
        'quiet': True,
        'no_warnings': True
    }

    valid_urls = []
    excluded_urls = []

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for idx, url in enumerate(urls, start=1):
            try:
                info = ydl.extract_info(url, download=False)
                title = info.get('title', '제목 없음')
                description = info.get('description', '')

                # 2. 필터링 조건
                # (1) '가게'라는 말이나 '장소'라는 말이 없거나
                # (2) 'china' (대소문자 무관)가 포함되어 있으면 제외
                if (('가게' not in description) and ('장소' not in description)) or ('china' in description.lower()):
                    print(f"[{idx}/{len(urls)}] [제외] {title}")
                    excluded_urls.append(url)
                else:
                    print(f"[{idx}/{len(urls)}] [유지] {title}")
                    valid_urls.append(url)

            except Exception as e:
                print(f"[{idx}/{len(urls)}] [오류/제외] {url} - {e}")
                excluded_urls.append(url)

    # 3-1. 통과된 URL 목록 저장
    with open(valid_output_file, "w", encoding="utf-8") as f:
        for valid_url in valid_urls:
            f.write(valid_url + "\n")

    # 3-2. 제외된 URL 목록 저장
    with open(excluded_output_file, "w", encoding="utf-8") as f:
        for excluded_url in excluded_urls:
            f.write(excluded_url + "\n")

    print(f"\n처리 완료!")
    print(f"- 통과된 URL: {len(valid_urls)}개 -> '{valid_output_file}'")
    print(f"- 제외된 URL: {len(excluded_urls)}개 -> '{excluded_output_file}'")

총 77개의 URL을 확인합니다...

[1/77] [유지] (연휴특집) 정식당 출신 셰프 식당! 뉴욕을 접수한 한식의 힘⭐ | Jua
[2/77] [유지] 대체 얼마나 맛있길래? 줄 서서 먹는 닭백숙 맛집🐔 | 청송 원조신촌식당 | 10년만에 또간집🙏🏻
[3/77] [제외] 서울 강북 5성급 호텔&조식 총정리⭐이번 연휴 호캉스는 '이곳'으로 가세요
[4/77] [제외] 서울 강남 5성급 호텔 총정리, 어디가 제일 좋을까요? 내돈내산으로 리뷰해 봤습니다!
[5/77] [유지] 뉴욕 오픈 1년 만에 미쉐린 2스타? 현지인들도 난리 난 한식당 근황 | Joo Ok
[6/77] [유지] 우리나라 Top 셰프들이 감탄한 정관스님 제자 '20대 셰프' 식당 | 평창 예미헌
[7/77] [유지] 독일 최고의 레스토랑? 꾸준히 미슐랭 3스타 유지 중! | Waldhotel Sonnora
[8/77] [유지] 뉴욕을 대표하는 코리안 파인다이닝 | Atomix | 미슐랭 2스타
[9/77] [유지] 나폴리맛피아 신상 레스토랑 솔직 리뷰합니다. | 베수비오 | 권성준 셰프
[10/77] [유지] 화제의 윤남노 셰프 신상 식당⭐ 다녀온 진짜 솔직 후기 | 비스트로노미 노뜨르
[11/77] [유지] 뮌헨을 대표하는 와인리스트 좋은 파인다이닝 | Restaurant Alois - Dallmayr Fine Dining
[12/77] [유지] 구글 평점 떡상해서, 줄 서는 피렌체 로컬 식당 | Osteria del Cinghiale bianco
[13/77] [유지] 비현실적인 테니스장 호텔...와인 가격조차 말이 안됨 | 포지타노 Il San Pietro di Positano
[14/77] [유지] 미친 가성비(?) 와인🍷과 역대급 테니스장!🎾 근데 왜 미쉐린 1키일까? | 토스카나 Hotel Il Pellicano
[15/77] [유지] 남프랑스 최고의 미슐랭 3스타 | 아름다운 지중해 풍경은 서비스✨ | La Villa Madie
[16/77] [유지] 서울에서 제일 맛있는 목살?

### 영상 스크립트 추출

In [ ]:
# import concurrent.futures
# import os
# import re
# import time
# from playwright.sync_api import sync_playwright


# def get_video_id(url):
#     match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11})', url)
#     return match.group(1) if match else url


# def save_youtube_transcript_browser(page, video_url, output_filename):
#     video_id = get_video_id(video_url)
#     clean_url = f'https://www.youtube.com/watch?v={video_id}'

#     try:
#         # 1. 페이지 접속
#         page.goto(clean_url, wait_until='domcontentloaded', timeout=20000)
#         time.sleep(2)

#         # 2. '...더보기' 클릭
#         expand_btn = page.query_selector(
#             '#expand, tp-yt-paper-button#expand, #description-inline-expander'
#         )
#         if expand_btn:
#             try:
#                 expand_btn.click()
#                 time.sleep(0.5)
#             except:
#                 pass

#         # 3. '스크립트 표시' 버튼 클릭
#         transcript_btn = page.query_selector(
#             'button[aria-label*="스크립트"], button[aria-label*="Transcript"], ytd-video-description-transcript-section-renderer button'
#         )

#         if not transcript_btn:
#             for b in page.query_selector_all('button'):
#                 t = b.inner_text()
#                 if '스크립트 표시' in t or 'Show transcript' in t or '스크립트' in t:
#                     transcript_btn = b
#                     break

#         if not transcript_btn:
#             print(f'⚠️ 자막/스크립트 버튼을 찾을 수 없습니다: {clean_url}')
#             return False

#         transcript_btn.click()
#         time.sleep(1.5)

#         # 4. 자막 추출
#         page.wait_for_selector(
#             'ytd-transcript-segment-renderer', timeout=8000
#         )
#         segments = page.query_selector_all('ytd-transcript-segment-renderer')

#         lines = []
#         for seg in segments:
#             text = seg.inner_text().strip()
#             parts = text.split('\n')
#             if len(parts) >= 2:
#                 lines.append(parts[-1].strip())
#             elif text:
#                 lines.append(text)

#         if not lines:
#             print(f'⚠️ 자막 내용이 비어 있습니다: {clean_url}')
#             return False

#         # 5. 파일 저장
#         with open(output_filename, 'w', encoding='utf-8') as f:
#             f.write('\n'.join(lines))

#         print(f' 성공적으로 저장되었습니다: {output_filename}')
#         return True

#     except Exception as e:
#         print(f' 자막 추출 실패 ({video_id}): {e}')
#         return False


# def _run_worker(input_file, output_dir):
#     """별도 독립 스레드에서 Playwright 실행 (Windows Jupyter 충돌 방지)"""
#     os.makedirs(output_dir, exist_ok=True)
#     if not os.path.exists(input_file):
#         print(f"'{input_file}' 파일을 찾을 수 없습니다.")
#         return

#     with open(input_file, 'r', encoding='utf-8') as f:
#         urls = [line.strip() for line in f if line.strip()]

#     print(
#         f'총 {len(urls)}개의 URL 브라우저 자동 수집을 시작합니다.\n'
#         + '-' * 50
#     )
#     success_count = 0

#     with sync_playwright() as p:
#         browser = p.chromium.launch(
#             headless=True,
#             args=[
#                 '--lang=ko-KR,ko',
#                 '--disable-blink-features=AutomationControlled',
#             ],
#         )
#         context = browser.new_context(
#             user_agent=(
#                 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
#                 ' AppleWebKit/537.36 (KHTML, like Gecko)'
#                 ' Chrome/123.0.0.0 Safari/537.36'
#             ),
#             locale='ko-KR',
#         )
#         page = context.new_page()

#         for idx, url in enumerate(urls, 1):
#             video_id = get_video_id(url)
#             output_filename = os.path.join(output_dir, f'{video_id}.txt')
#             print(f'[{idx}/{len(urls)}] 처리 중: {url}')

#             if save_youtube_transcript_browser(page, url, output_filename):
#                 success_count += 1

#             time.sleep(1.0)

#         browser.close()

#     print('-' * 50)
#     print(f'✨ 작업 완료: 총 {len(urls)}개 중 {success_count}개 성공')


# def process_urls_from_file(
#     input_file='../data/urls/filtered_channel_video_urls.txt', output_dir='../data/transcripts'
# ):
#     with concurrent.futures.ThreadPoolExecutor() as executor:
#         future = executor.submit(_run_worker, input_file, output_dir)
#         future.result()

Writing get_transcripts.py


In [ ]:
import re
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig
import os


def get_video_id(url):
    # 유튜브 URL에서 Video ID 추출
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11})', url)
    return match.group(1) if match else url


def save_youtube_transcript(video_url, output_filename='transcript.txt'):
    video_id = get_video_id(video_url)

    try:
        # 한국어('ko') 자막 우선, 없으면 영어('en') 자막 가져오기
        # yt = YouTubeTranscriptApi()
        
        ytt_api = YouTubeTranscriptApi(
            proxy_config=WebshareProxyConfig(
                proxy_username="<proxy-username>",
                proxy_password="<proxy-password>",
            )
        )
        transcript = ytt_api.fetch(video_id, languages=['ko', 'en'])

        # 텍스트 추출 및 파일 저장
        with open(output_filename, 'w', encoding='utf-8') as f:
            for entry in transcript:
                # 1) 시간 정보 없이 텍스트만 저장할 때 (.text 사용)
                f.write(f"{entry.text}\n")

                # 2) 만약 타임스탬프 포함을 원하면 아래 주석 해제 (.start 사용)
                # start_time = int(entry.start)
                # f.write(f"[{start_time//60:02d}:{start_time%60:02d}] {entry.text}\n")

        print(f" 성공적으로 저장되었습니다: {output_filename}")

    except Exception as e:
        print(f" 스크립트를 가져오는 중 오류 발생: {e}")


In [2]:
def process_urls_from_file(
    input_file='../data/urls/filtered_channel_video_urls.txt', output_dir='../data/transcripts'
):
    """URL 목록 파일에서 한 줄씩 읽어 개별 파일로 저장"""
    os.makedirs(output_dir, exist_ok=True)
    if not os.path.exists(input_file):
        print(f"'{input_file}' 파일을 찾을 수 없습니다.")
        return
    with open(input_file, 'r', encoding='utf-8') as f:
        urls = [line.strip() for line in f if line.strip()]
    print(f"총 {len(urls)}개의 URL 작업을 시작합니다.\n" + '-' * 50)
    success_count = 0
    for idx, url in enumerate(urls, 1):
        video_id = get_video_id(url)
        output_filename = os.path.join(output_dir, f'{video_id}.txt')
        print(f"[{idx}/{len(urls)}] 처리 중: {url}")
        if save_youtube_transcript(url, output_filename):
            success_count += 1
    print('-' * 50)
    print(f"✨ 작업 완료: 총 {len(urls)}개 중 {success_count}개 성공")


In [4]:
import os

process_urls_from_file(
    input_file='../data/urls/filtered_channel_video_urls.txt'
)

총 72개의 URL 작업을 시작합니다.
--------------------------------------------------
[1/72] 처리 중: https://www.youtube.com/watch?v=vsjDwPCJi8I
 스크립트를 가져오는 중 오류 발생: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=vsjDwPCJi8I (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))
[2/72] 처리 중: https://www.youtube.com/watch?v=YeZJduJmo0Y
 스크립트를 가져오는 중 오류 발생: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=YeZJduJmo0Y (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))
[3/72] 처리 중: https://www.youtube.com/watch?v=LQzb2LKemY4


KeyboardInterrupt: 

In [16]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# transcripts 폴더 안의 모든 .txt 파일을 LangChain Document 객체로 로드
loader = DirectoryLoader(
    path='../data/transcripts',
    glob='*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
)

docs = loader.load()
print(f"총 {len(docs)}개의 Document가 로드되었습니다.")



총 52개의 Document가 로드되었습니다.


In [108]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

load_dotenv()

llm = init_chat_model(
    model='openai:gpt-4o-mini',  # 또는 openai:gpt-4.1-mini
    temperature=0.1             # 정보 누락 방지 및 정확도를 위해 낮은 온도로 설정
)

# 1. 시스템 프롬프트: 추출 항목 및 STT 오타 보정 지침 명시
system_prompt = """
너는 미식/맛집 전문 데이터 분석가야.
제공되는 텍스트는 유튜브 영상의 자막(STT 스크립트)으로, 오타나 부정확한 음성 인식이 포함되어 있을 수 있어.
문맥을 파악하여 오타를 자연스럽게 보정하면서, **식당과 음식에 관한 모든 디테일(메뉴, 맛 평가, 조리법, 특징 등)**을 최대한 누락 없이 추출해줘.

아래의 구조화된 서식에 맞춰 상세히 작성해줘:

---
# 🏪 1. 식당 기본 정보
- **상호명**: (문맥상 추정되는 정확한 가게 이름)
- **위치/지역**: (언급된 동네, 랜드마크, 상권 등)
- **식당 종류/컨셉**: (예: 중식당, 짬뽕 전문점, 캐주얼 다이닝 등)
- **식당 특징 및 분위기**: (그릇/식기, 인테리어, 주문 방식, 운영 방식 등 언급된 모든 특징)

# 🍲 2. 주문/언급된 메뉴 상세 분석 (언급된 모든 메뉴별로 작성)
*(여러 메뉴가 등장하면 각 메뉴별로 아래 항목을 반복해서 작성)*
- **메뉴명**:
- **조리 특징 / 식재료 디테일**: (예: 쌀 종류(자스민 라이스), 튀김 방식, 소스 배합, 육수 등)
- **맛 및 식감 평가**: (바삭함, 매운맛, 향, 간, 유튜버가 표현한 구체적인 미식 표현)
- **먹는 방법 / 유튜버의 추천 팁**: (찍먹/부먹, 소스 조합, 곁들임 등)

# 🍶 3. 주류 / 음료 정보 (언급된 경우)
- **주류명/종류**: (예: 백주/고량주 - 검남춘 등, 도수, 향의 종류 등)
- **페어링 및 평가**: (음식과의 궁합, 유튜버의 시음 평)

# 💡 4. 이 식당만의 특별한 차별점 / 꿀팁 (Key Point)
- (다른 일반 식당과 다른 점, 숨겨진 팁, 포장/방문 시 참고사항 등)

# 📝 5. 유튜버의 최종 총평 및 한줄평
- (총평 요약)
---
"""

result = []

# 각 문서의 내용 및 메타데이터 확인
for doc in docs[:2]:  # 처음 3개만 확인
    print(f"Source: {doc.metadata['source']}")
    print(f"내용 미리보기: {doc.page_content[:100]}...\n")
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"다음 유튜브 영상 자막 스크립트에서 식당과 음식 관련 정보를 빠짐없이 최대한 자세히 추출해줘:\n\n{doc}")
    ]

    response = llm.invoke(messages)

    result.append(response.content)

    print(f'요약: {response.content}')

# 2. 파일 읽기 및 LLM 호출
# with open("test.txt", "r", encoding="utf-8") as f:
#     script_content = f.read()



Source: transcripts\3vYwR8V8IaU.txt
내용 미리보기: 안녕하세요 비밀이야 입니다
오늘 소개해 드릴 곳은...
파리 리옹역 인근에 위치한 작은 식당
자 저는 파리 가리 드 리옹(Gare de Lyon)역 앞에 있는
Le Quincy라는...

요약: ---
# 🏪 1. 식당 기본 정보
- **상호명**: Le Quincy
- **위치/지역**: 파리 리옹역(Gare de Lyon) 인근
- **식당 종류/컨셉**: 클래식하고 캐주얼한 프랑스 레스토랑
- **식당 특징 및 분위기**: 전통적인 프랑스 음식을 제공하며, 소박하고 아늑한 분위기. 주방을 지나가야 화장실에 갈 수 있는 독특한 구조. 직원이 불어를 구사하며, 영어가 통하지 않아도 큰 문제가 없는 서비스.

# 🍲 2. 주문/언급된 메뉴 상세 분석
- **메뉴명**: Saucisson (소시송)
  - **조리 특징 / 식재료 디테일**: 전통적인 프랑스 소시지, 간이 적절함.
  - **맛 및 식감 평가**: 진짜 맛있고, 간이 짜지 않음. 유튜버는 "진짜 진짜 진짜 맛있어요!"라고 표현.
  - **먹는 방법 / 유튜버의 추천 팁**: 소시송 모둠을 시키면 좋음.

- **메뉴명**: Foie gras de canard (오리 푸아그라)
  - **조리 특징 / 식재료 디테일**: 전통적인 오리 푸아그라.
  - **맛 및 식감 평가**: 비주얼이 훌륭하고, 맛이 뛰어남.
  - **먹는 방법 / 유튜버의 추천 팁**: 소시송과 함께 먹는 것이 좋음.

- **메뉴명**: Escargot (달팽이)
  - **조리 특징 / 식재료 디테일**: 보통 6개 또는 12개로 제공됨.
  - **맛 및 식감 평가**: 뜨겁고, 빵과 함께 먹으면 좋음.
  - **먹는 방법 / 유튜버의 추천 팁**: 조심해서 먹어야 하며, 빵과 함께 먹는 것을 추천.

- **메뉴명**: Tête de veau (송아지 머릿고기)
  - **조리 특징 / 식재료 디테일**: 프랑스식 송아지 머릿고기 스

In [111]:
from pprint import pprint

print(result[1])

---
# 🏪 1. 식당 기본 정보
- **상호명**: 호텔 일펠리카노 (Il Pellicano)
- **위치/지역**: 토스카나 남부, 이탈리아
- **식당 종류/컨셉**: 캐주얼 레스토랑, 그릴 요리 전문
- **식당 특징 및 분위기**: 호텔 내에 위치하며, 두 개의 레스토랑이 있음. 하나는 미쉐린 원스타의 파인다이닝 이탈리안 레스토랑이고, 다른 하나는 캐주얼한 분위기의 그릴 요리 레스토랑. 와인 리스트가 풍부하고 가격대가 높음. 테니스 코트와 바다 전망이 좋은 럭셔리 호텔.

# 🍲 2. 주문/언급된 메뉴 상세 분석
- **메뉴명**: 클럽 샌드위치 프로슈토 멜론
  - **조리 특징 / 식재료 디테일**: 프로슈토와 멜론을 사용. 멜론은 딱딱하고 달지 않은 품종.
  - **맛 및 식감 평가**: 프로슈토의 짭짤함과 멜론의 아삭한 식감이 조화를 이룸.
  - **먹는 방법 / 유튜버의 추천 팁**: 간단하게 즐길 수 있는 메뉴로, 가벼운 스낵으로 추천.

- **메뉴명**: 티본 스테이크
  - **조리 특징 / 식재료 디테일**: 고기 종류는 티본 스테이크로, 그릴에서 조리됨.
  - **맛 및 식감 평가**: 고기의 육즙이 풍부하고, 적절한 숙성으로 맛이 좋음. 그러나 조리 상태가 아쉬운 부분도 있음.
  - **먹는 방법 / 유튜버의 추천 팁**: 고기와 함께 그릴 베지터블을 곁들여 먹는 것을 추천.

- **메뉴명**: 멧돼지 라구 파스타
  - **조리 특징 / 식재료 디테일**: 멧돼지 고기를 사용한 라구 소스와 파스타.
  - **맛 및 식감 평가**: 깊은 풍미와 함께 고기의 식감이 잘 어우러짐.
  - **먹는 방법 / 유튜버의 추천 팁**: 파스타와 함께 와인을 곁들여 먹는 것을 추천.

- **메뉴명**: 봉골레
  - **조리 특징 / 식재료 디테일**: 조개를 사용한 파스타.
  - **맛 및 식감 평가**: 해산물의 신선함이 느껴지며, 간이 잘 맞음.
  - **먹는 방법 / 유튜버의 추천 팁**: 화이트 와인과 함께 즐기면 

In [4]:
import os
import yt_dlp
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field
from openai import OpenAI

# 1. .env 파일의 OPENAI_API_KEY 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# 2. Pydantic 추출 스키마 정의
class StoreInfo(BaseModel):
    name: str = Field(description="가게/식당/상호명 (예: 을지로보석, 멘야하나비)")
    address: str = Field(description="가게의 전체 주소 (도로명 또는 지번 주소. 상세 주소가 없으면 파악 가능한 시/구/동 단위)")
    extra_info: Optional[str] = Field(default=None, description="대표 메뉴, 타임스탬프, 한줄 특징 등 추가 정보가 있다면 기재")

class StoreExtractionResult(BaseModel):
    has_store_info: bool = Field(description="본문에 가게/장소 정보가 포함되어 있는지 여부")
    stores: List[StoreInfo] = Field(default_factory=list, description="추출된 가게 정보 목록 (단일 또는 복수)")


# 3. yt-dlp로 유튜브 설명 가져오는 함수
def get_youtube_description(url: str) -> dict:
    ydl_opts = {
        'skip_download': True,
        'quiet': True,
        'no_warnings': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title', ''),
            'description': info.get('description', '')
        }


# 4. OpenAI Structured Outputs로 가게 정보 추출하는 함수
def extract_stores_from_description(title: str, description: str) -> StoreExtractionResult:
    # 설명이 비어 있는 경우 처리
    if not description.strip():
        return StoreExtractionResult(has_store_info=False, stores=[])

    prompt = f"""
다음은 유튜브 영상의 제목과 상세 설명(더보기) 내용입니다.
본문에 소개된 맛집/가게/장소의 '가게명'과 '주소'를 모두 추출해주세요.
- 1개 영상에 여러 가게가 등장할 경우 모든 가게를 각각 분리하여 목록으로 추출하세요.
- 가게 정보가 없는 영상(단순 일상, 제품 리뷰 등)인 경우 has_store_info를 false로 설정하세요.

[영상 제목]: {title}
[영상 설명]:
{description}
"""

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",  # 비용 효율적이고 빠른 gpt-4o-mini 추천 (또는 gpt-4o)
        messages=[
            {"role": "system", "content": "너는 텍스트에서 장소 및 가게 정보(상호명, 주소)를 정확하게 구조화하여 추출하는 AI 비서야."},
            {"role": "user", "content": prompt}
        ],
        response_format=StoreExtractionResult,
    )

    return response.choices[0].message.parsed


# 5. 실행 테스트
if __name__ == "__main__":
    test_url = "https://www.youtube.com/watch?v=VnZAib60fGo"
    
    print(f"1. 유튜브 정보 가져오는 중... ({test_url})")
    video_data = get_youtube_description(test_url)
    print(f"영상 제목: {video_data['title']}\n")
    
    print("2. 가게 정보 스키마 추출 중...")
    extracted_data = extract_stores_from_description(
        title=video_data['title'],
        description=video_data['description']
    )
    
    print("\n=== 추출 결과 ===")
    print(f"가게 정보 포함 여부: {extracted_data.has_store_info}")
    print(f"추출된 가게 개수: {len(extracted_data.stores)}개\n")
    
    for idx, store in enumerate(extracted_data.stores, 1):
        print(f"[{idx}] 가게명 : {store.name}")
        print(f"    주소   : {store.address}")
        if store.extra_info:
            print(f"    비고   : {store.extra_info}")
        print("-" * 40)

1. 유튜브 정보 가져오는 중... (https://www.youtube.com/watch?v=VnZAib60fGo)
영상 제목: 라멘 먹으러 일본 왔습니다🍜삿포로 라멘 투어 | 멘야 사이미, 토쿠이치

2. 가게 정보 스키마 추출 중...

=== 추출 결과 ===
가게 정보 포함 여부: True
추출된 가게 개수: 2개

[1] 가게명 : 멘야 사이미
    주소   : 5 Chome-3-12 Misono 10 Jo, Toyohira Ward, Sapporo, Hokkaido 062-0010 일본
----------------------------------------
[2] 가게명 : 토쿠이치
    주소   : 일본 060-0011 Hokkaido, Sapporo, Chuo Ward, Kita 11 Jonishi, 22 Chome−1−26 1F
----------------------------------------


In [104]:
# url -> google places api 불러오기

# 1. Pydantic 스키마 정의 (상호명, 주소, 국가코드)
# 2. [Step 1] yt-dlp 유튜브 (title, description) 추출 
# 3. [Step 2] OpenAI description, 가게 정보 스키마 추출
# 4. [Step 3] Google Places API (New)로 가게 상세 정보 조회
#------------------------------------------------------------

import os
import json
import requests
import yt_dlp
from dotenv import load_dotenv
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
from openai import OpenAI

# -------------------------------------------------------------
# 0. 환경 변수 로드 (.env)
# -------------------------------------------------------------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("⚠️ .env 파일에 OPENAI_API_KEY가 필요합니다.")
if not GOOGLE_PLACES_API_KEY:
    print("⚠️ 경고: .env 파일에 GOOGLE_PLACES_API_KEY가 없습니다. Google Places 검색 단계에서 오류가 날 수 있습니다.")

client = OpenAI(api_key=OPENAI_API_KEY)


# -------------------------------------------------------------
# 1. Pydantic 스키마 정의 (상호명, 주소, 국가코드)
# -------------------------------------------------------------
class StoreInfo(BaseModel):
    name: str = Field(description="가게/식당/상호명 (예: 산청숯불가든, 이치란, Joe's Pizza)")
    address: str = Field(description="가게의 전체 주소 또는 도시/지역명")
    country_code: str = Field(
        default="KR",
        description=(
            "가게 위치 국가의 ISO 3166-1 alpha-2 2자리 국가 코드 (대문자). "
            "예: 한국 KR, 일본 JP, 미국 US, 태국 TH, 베트남 VN, 대만 TW, 프랑스 FR 등. "
            "주소/도시명/본문 맥락으로 판단하고 불분명하면 'KR'"
        )
    )
    extra_info: Optional[str] = Field(default=None, description="메뉴, 타임스탬프, 한줄 특징 등 추가 정보")

class StoreExtractionResult(BaseModel):
    has_store_info: bool = Field(description="본문에 가게/음식점 정보가 포함되어 있는지 여부")
    stores: List[StoreInfo] = Field(default_factory=list, description="추출된 가게 정보 목록")


# -------------------------------------------------------------
# 2. [Step 1] yt-dlp 유튜브 (title, description) 추출 
# -------------------------------------------------------------
def get_youtube_description(url: str) -> dict:
    ydl_opts = {
        'skip_download': True,
        'quiet': True,
        'no_warnings': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title', ''),
            'description': info.get('description', '')
        }


# -------------------------------------------------------------
# 3. [Step 2] OpenAI description, 가게 정보 스키마 추출
# -------------------------------------------------------------
def extract_stores_from_description(title: str, description: str) -> StoreExtractionResult:
    if not description.strip():
        return StoreExtractionResult(has_store_info=False, stores=[])

    prompt = f"""
다음은 유튜브 영상의 제목과 설명(더보기) 내용입니다.
본문에 소개된 음식점/가게의 '가게명', '주소', 그리고 해당 국가의 '2자리 국가코드(KR, JP, US 등)'를 추출해주세요.
- 여러 가게가 등장하면 각각 분리하여 목록으로 추출하세요.
- 가게 정보가 전혀 없는 경우 has_store_info를 false로 설정하세요.

[영상 제목]: {title}
[영상 설명]:
{description}
"""

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 텍스트에서 상호명, 주소, 국가코드를 정확하게 구조화하여 추출하는 전문가야."},
            {"role": "user", "content": prompt}
        ],
        response_format=StoreExtractionResult,
    )
    return response.choices[0].message.parsed


# -------------------------------------------------------------
# 4. [Step 3] Google Places API (New)로 가게 상세 정보 조회
# -------------------------------------------------------------
def search_restaurant_google(
    store_name: str, 
    address: Optional[str] = None, 
    country_code: Optional[str] = "KR"
) -> Optional[Dict[str, Any]]:
    if not GOOGLE_PLACES_API_KEY:
        return None
    
    # 쿼리: 가게명 + 주소 결합 (동명 가게 혼동 방지)
    query = f"{store_name} {address}".strip() if address else store_name
    url = "https://places.googleapis.com/v1/places:searchText"
    
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.formattedAddress,"
            "places.rating,"
            "places.userRatingCount,"
            "places.location,"
            "places.googleMapsUri,"
            "places.internationalPhoneNumber,"
            "places.nationalPhoneNumber,"
            "places.primaryTypeDisplayName,"
            "places.businessStatus"
        )
    }
    
    payload = {
        "textQuery": query,
        "languageCode": "ko",
        "maxResultCount": 1
    }
    
    # 추출된 국가 코드를 regionCode에 적용
    if country_code:
        payload["regionCode"] = country_code.upper().strip()
    
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        print(f"API 호출 오류 ({response.status_code}): {response.text}")
        return None
        
    data = response.json()
    places = data.get("places", [])
    if not places:
        return None
    
    place = places[0]
    return {
        "place_id": place.get("id"),
        "google_name": place.get("displayName", {}).get("text"),
        "formatted_address": place.get("formattedAddress"),
        "rating": place.get("rating"),
        "user_rating_count": place.get("userRatingCount"),
        "category": place.get("primaryTypeDisplayName", {}).get("text"),
        "latitude": place.get("location", {}).get("latitude"),
        "longitude": place.get("location", {}).get("longitude"),
        "phone": place.get("internationalPhoneNumber") or place.get("nationalPhoneNumber"),
        "google_maps_url": place.get("googleMapsUri"),
        "business_status": place.get("businessStatus")
    }


# -------------------------------------------------------------
# 5. 전체 파이프라인 실행 (URL: https://www.youtube.com/watch?v=VnZAib60fGo)
# -------------------------------------------------------------
target_url = "https://www.youtube.com/watch?v=VnZAib60fGo"

print("=" * 60)
print(f"🚀 [STEP 1] 유튜브 설명 가져오는 중...\nURL: {target_url}")
video_data = get_youtube_description(target_url)
print(f"🎬 영상 제목: {video_data['title']}")
print(f"📄 설명 길이: {len(video_data['description'])} 글자")
print("=" * 60)

print("\n🤖 [STEP 2] LLM으로 가게 정보 & 국가코드 스키마 추출 중...")
extracted = extract_stores_from_description(video_data['title'], video_data['description'])
print(f"📌 가게 정보 포함 여부: {extracted.has_store_info}")
print(f"📌 추출된 가게 개수: {len(extracted.stores)}개\n")

print("=" * 60)
print("📍 [STEP 3] Google Places API 조회 결과")
print("=" * 60)

for idx, store in enumerate(extracted.stores, 1):
    print(f"\n--- [{idx}] {store.name} ---")
    print(f"• [LLM 추출 주소]    : {store.address}")
    print(f"• [LLM 감지 국가]    : {store.country_code}")
    if store.extra_info:
        print(f"• [비고/메뉴]        : {store.extra_info}")
    
    # Google Places 검색 실행
    google_res = search_restaurant_google(
        store_name=store.name,
        address=store.address,
        country_code=store.country_code
    )
    
    if google_res:
        print(f"  👉 [구글 정식 상호] : {google_res['google_name']}")
        print(f"  👉 [구글 표준 주소] : {google_res['formatted_address']}")
        print(f"  👉 [카테고리]       : {google_res['category']}")
        print(f"  👉 [평점 / 리뷰]    : ⭐️ {google_res['rating']} ({google_res['user_rating_count']}개 리뷰)")
        print(f"  👉 [위도 / 경도]    : {google_res['latitude']}, {google_res['longitude']}")
        print(f"  👉 [구글 지도 URL]  : {google_res['google_maps_url']}")
    else:
        print("  ❌ Google Places 검색 결과가 없습니다.")

⚠️ 경고: .env 파일에 GOOGLE_PLACES_API_KEY가 없습니다. Google Places 검색 단계에서 오류가 날 수 있습니다.
🚀 [STEP 1] 유튜브 설명 가져오는 중...
URL: https://www.youtube.com/watch?v=VnZAib60fGo
🎬 영상 제목: 라멘 먹으러 일본 왔습니다🍜삿포로 라멘 투어 | 멘야 사이미, 토쿠이치
📄 설명 길이: 426 글자

🤖 [STEP 2] LLM으로 가게 정보 & 국가코드 스키마 추출 중...
📌 가게 정보 포함 여부: True
📌 추출된 가게 개수: 2개

📍 [STEP 3] Google Places API 조회 결과

--- [1] 멘야 사이미 ---
• [LLM 추출 주소]    : 5 Chome-3-12 Misono 10 Jo, Toyohira Ward, Sapporo, Hokkaido 062-0010
• [LLM 감지 국가]    : JP
  ❌ Google Places 검색 결과가 없습니다.

--- [2] 토쿠이치 ---
• [LLM 추출 주소]    : 일본 060-0011 Hokkaido, Sapporo, Chuo Ward, Kita 11 Jonishi, 22 Chome−1−26 1F
• [LLM 감지 국가]    : JP
  ❌ Google Places 검색 결과가 없습니다.


In [ ]:
# google places api 에서 '현지 상호명' 서치 -> 미슐랭 공홈 '정식 상호명' 서치 -> LLM으로 등급 판별 (정규 5년 / 특별판 2회)

import os
import json
import requests
import yt_dlp
from enum import Enum
from dotenv import load_dotenv
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
from openai import OpenAI

# -------------------------------------------------------------
# 0. 환경 변수 로드
# -------------------------------------------------------------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("⚠️ .env 파일에 OPENAI_API_KEY가 필요합니다.")

client = OpenAI(api_key=OPENAI_API_KEY)


# -------------------------------------------------------------
# 1. Pydantic 스키마 정의
# -------------------------------------------------------------
class EditionType(str, Enum):
    REGULAR = "REGULAR"    # 정규 연례 가이드 (서울, 도쿄, 파리 등)
    SPECIAL = "SPECIAL"    # 비정기 특별판 (홋카이도, 규슈 등)
    NONE = "NONE"          # 미등재

class MichelinGrade(str, Enum):
    THREE_STARS = "3_STARS"
    TWO_STARS = "2_STARS"
    ONE_STAR = "1_STAR"
    BIB_GOURMAND = "BIB_GOURMAND"
    SELECTED = "SELECTED"
    NONE = "NONE"

class MichelinYearRecord(BaseModel):
    edition: str = Field(description="연도 또는 에디션명 (예: '2025', '2017 홋카이도 특별판')")
    grade: MichelinGrade = Field(description="획득 등급")

class MichelinInfo(BaseModel):
    is_michelin: bool = Field(description="미슐랭 등재 이력이 있는지 여부")
    edition_type: EditionType = Field(description="가이드 종류 (REGULAR, SPECIAL, NONE)")
    latest_grade: MichelinGrade = Field(description="가장 최근 획득한 등급")
    history: List[MichelinYearRecord] = Field(
        default_factory=list,
        description="정규 도시: 최근 5개년(2021~2025) 이력 / 특별판: 최근 2회 에디션 이력"
    )
    is_active: bool = Field(description="현재 유효 여부 (정규는 최신년도 유지, 특별판은 마지막 특별판 등재)")
    summary_badge: str = Field(description="요약 뱃지 (예: '2012·2017 홋카이도 특별판 2회 연속 빕구르망', '2021-2025 5년 연속 2스타', '미등재')")
    michelin_url: Optional[str] = Field(default=None, description="공식 링크")
    reason: str = Field(description="판정 상세 근거")

class StoreInfo(BaseModel):
    korean_name: str = Field(description="영상 본문에 언급된 한국어 상호명 (예: 멘야 사이미, 밍글스)")
    address: str = Field(description="가게 주소 또는 도시/지역명")
    country_code: str = Field(default="KR", description="ISO 2자리 국가코드 (대문자: KR, JP, US, FR 등)")
    extra_info: Optional[str] = Field(default=None, description="메뉴, 타임스탬프 등")

class StoreExtractionResult(BaseModel):
    has_store_info: bool = Field(description="가게 정보 포함 여부")
    stores: List[StoreInfo] = Field(default_factory=list, description="추출된 가게 목록")


# -------------------------------------------------------------
# 2. [Step 1] yt-dlp: 유튜브 설명 가져오기
# -------------------------------------------------------------
def get_youtube_description(url: str) -> dict:
    ydl_opts = {'skip_download': True, 'quiet': True, 'no_warnings': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return {'title': info.get('title', ''), 'description': info.get('description', '')}


# -------------------------------------------------------------
# 3. [Step 2] OpenAI: 한국 상호명, 주소, 국가코드 추출
# -------------------------------------------------------------
def extract_stores_from_description(title: str, description: str) -> StoreExtractionResult:
    if not description.strip():
        return StoreExtractionResult(has_store_info=False, stores=[])

    prompt = f"""
다음 유튜브 영상의 제목과 설명에서 소개된 음식점의 '한국어 상호명(korean_name)', '주소(address)', '2자리 국가코드(country_code: KR, JP, US, FR 등)'를 추출하세요.
- 여러 가게가 있으면 각각 분리하여 목록으로 만드세요.
- 가게 정보가 없으면 has_store_info를 false로 설정하세요.

[영상 제목]: {title}
[영상 설명]:
{description}
"""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 상호명, 주소, 국가코드를 정확하게 구조화하여 추출하는 전문가야."},
            {"role": "user", "content": prompt}
        ],
        response_format=StoreExtractionResult,
    )
    return response.choices[0].message.parsed


# -------------------------------------------------------------
# 4. [Step 3] Google Places API: 해당 국가의 '공식 현지 상호명' 획득
# -------------------------------------------------------------
def search_restaurant_google(korean_name: str, address: Optional[str] = None, country_code: str = "KR") -> Optional[Dict[str, Any]]:
    if not GOOGLE_PLACES_API_KEY:
        return None

    query = f"{korean_name} {address}".strip() if address else korean_name
    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
        "X-Goog-FieldMask": (
            "places.id,places.displayName,places.formattedAddress,"
            "places.rating,places.userRatingCount,places.location,"
            "places.googleMapsUri,places.primaryTypeDisplayName"
        )
    }

    payload = {
        "textQuery": query,
        "maxResultCount": 1
        # languageCode를 지정하지 않음으로써 Google이 해당 장소의 '현지 공식 원어 명칭(예: 麺屋 彩未)'을 반환하도록 유도
    }
    if country_code:
        payload["regionCode"] = country_code.upper().strip()

    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        return None

    places = response.json().get("places", [])
    if not places:
        return None

    place = places[0]
    return {
        "place_id": place.get("id"),
        "official_local_name": place.get("displayName", {}).get("text"),  # 👈 Google이 보증하는 현지 정식 상호명
        "formatted_address": place.get("formattedAddress"),
        "rating": place.get("rating"),
        "user_rating_count": place.get("userRatingCount"),
        "category": place.get("primaryTypeDisplayName", {}).get("text"),
        "lat": place.get("location", {}).get("latitude"),
        "lng": place.get("location", {}).get("longitude"),
        "maps_url": place.get("googleMapsUri")
    }


# -------------------------------------------------------------
# 5. [Step 4] 미슐랭 판별: '구글 공식 상호명'으로 정확한 서치
# -------------------------------------------------------------
def check_michelin_status(official_name: str, korean_name: str, address: Optional[str] = None, country_code: str = "KR") -> MichelinInfo:
    if not TAVILY_API_KEY:
        return MichelinInfo(
            is_michelin=False, edition_type=EditionType.NONE,
            latest_grade=MichelinGrade.NONE, is_active=False,
            summary_badge="미등재", reason="TAVILY_API_KEY 없음"
        )

    # 1) Tavily 검색: 구글 공식 상호명을 메인 키워드로 검색
    query = f'"{official_name}" Michelin Guide 미슐랭 빕구르망 스타 특별판'
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "search_depth": "basic",
        "max_results": 5,
        "include_answer": True
    }

    context = ""
    try:
        res = requests.post(url, json=payload, timeout=10).json()
        results = res.get("results", [])

        # 검색 결과가 부족할 경우 한국 상호명과 주소를 묶어 2차 보완 검색
        if not results:
            payload["query"] = f'"{korean_name}" "{official_name}" "{address or ""}" 미슐랭 Michelin'
            res = requests.post(url, json=payload, timeout=10).json()
            results = res.get("results", [])

        if res.get("answer"):
            context += f"[AI 검색 요약]: {res.get('answer')}\n\n"
        context += "\n".join([f"출처: {r.get('title')}\nURL: {r.get('url')}\n내용: {r.get('content')}" for r in results])
    except Exception as e:
        context = f"검색 에러: {e}"

    # 2) LLM 정밀 이력 판별
    prompt = f"""
당신은 전 세계 미슐랭 가이드 레스토랑 데이터 분석 전문가입니다.
검색 결과를 분석하여 식당 [{official_name} / 한국명: {korean_name}] (주소: {address or '미지정'}, 국가: {country_code})의 미슐랭 등재 이력을 판별하세요.

[이력 수집 기준]
1. 정규 도시 (서울, 도쿄, 파리, 뉴욕 등):
   - 최근 5개년(2021 ~ 2025) 동안의 등재 이력을 history 리스트로 작성하세요.
   - 최신년도(2024/2025)에 유지 중이면 is_active = True

2. 비정기 특별판 지역 (삿포로/홋카이도, 규슈 등):
   - 최근 2회 특별판 에디션(예: 홋카이도 2012, 2017 특별판) 이력을 history 리스트로 작성하세요.
   - 마지막 특별판에 등재되었다면 (이후 새 가이드가 없으므로) is_active = True

3. 미등재 식당:
   - is_michelin = False, edition_type = NONE, latest_grade = NONE, summary_badge = "미등재"

[검색 데이터]:
{context if context else "검색 결과 없음"}
"""

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 미슐랭 가이드 공식 데이터 분석 AI야."},
            {"role": "user", "content": prompt}
        ],
        response_format=MichelinInfo
    )
    return response.choices[0].message.parsed


# -------------------------------------------------------------
# 6. 파이프라인 실행 테스트
# -------------------------------------------------------------
target_url = "https://www.youtube.com/watch?v=VnZAib60fGo"

print("=" * 70)
print(f"🎬 [STEP 1] 유튜브 설명 수집: {target_url}")
video_data = get_youtube_description(target_url)
print(f"📌 영상 제목: {video_data['title']}")
print("=" * 70)

print("\n🤖 [STEP 2] LLM 상호명, 주소, 국가코드 추출 중...")
extracted = extract_stores_from_description(video_data['title'], video_data['description'])
print(f"👉 총 {len(extracted.stores)}개 가게 추출 완료!\n")

print("=" * 70)
print("🗺️ [STEP 3 & 4] Google Places 공식 상호명 획득 ➡️ 미슐랭 이력 서치")
print("=" * 70)

for idx, store in enumerate(extracted.stores, 1):
    print(f"\n[{idx}] 🇰🇷 유튜브 추출 한국명 : {store.korean_name} (국가: {store.country_code})")
    print(f"    📍 본문 추출 주소       : {store.address}")

    # 3단계: Google Places로 해당 국가의 정식 상호명 획득
    g_info = search_restaurant_google(store.korean_name, store.address, store.country_code)

    if g_info:
        official_name = g_info['official_local_name']
        print(f"    🏛️ [구글 공식 현지명]  : {official_name}")
        print(f"    📍 [구글 표준 주소]    : {g_info['formatted_address']}")
        print(f"    ⭐️ [구글 평점/리뷰]    : {g_info['rating']}점 ({g_info['user_rating_count']}개 리뷰)")
        print(f"    🔗 [구글 지도 URL]     : {g_info['maps_url']}")
    else:
        official_name = store.korean_name
        print(f"    🏛️ [Google Places] ❌ 검색 실패 (한국 상호명으로 대체 검색)")

    # 4단계: Google 정식 명칭으로 미슐랭 가이드 정밀 서치
    m_info = check_michelin_status(
        official_name=official_name,
        korean_name=store.korean_name,
        address=g_info['formatted_address'] if g_info else store.address,
        country_code=store.country_code
    )

    print(f"\n    🏅 [미슐랭 판정 결과]")
    print(f"    • 🏷️ 요약 뱃지   : {m_info.summary_badge}")
    print(f"    • 📂 가이드 종류 : {m_info.edition_type.value}")
    print(f"    • 🎖️ 최신 등급   : {m_info.latest_grade.value}")
    print(f"    • ⚡ 현재 유효   : {'✅ 유효함' if m_info.is_active else '⚪ 만료/해당없음'}")

    if m_info.history:
        print("    • 📜 등재 이력   :")
        for h in m_info.history:
            print(f"        - {h.edition} : {h.grade.value}")

    print(f"    • 📝 판정 근거   : {m_info.reason}")
    print("-" * 70)